In [1]:
!pip install torch torchvision pillow matplotlib
import os
import random
import time
from datetime import datetime
from typing import Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, utils
from PIL import Image

In [12]:
DATA_DIR = "/kaggle/input/datasets/arahul22/a2-q1-blink/Blink Detection Dataset"
OUTPUT_DIR = "cgan_outputs"
IMAGE_SIZE = 64
BATCH_SIZE = 64
EPOCHS = 10
LR = 0.0002
BETAS = (0.5, 0.999)
Z_DIM = 100
NUM_CLASSES = 2  # open, closed
NUM_WORKERS = 2
SAMPLE_EVERY = 5
TRAIN_SPLIT = 0.9
SEED = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [ ]:
class BlinkDataset(Dataset):
    def __init__(self, root_dir: str, image_size: int = 64):
        self.samples = []
        self.class_to_idx = {"open": 0, "closed": 1}

        class_dirs = {}
        for entry in os.listdir(root_dir):
            entry_path = os.path.join(root_dir, entry)
            if not os.path.isdir(entry_path):
                continue
            key = entry.strip().lower()
            if key in self.class_to_idx:
                class_dirs[key] = entry_path

        for class_key, class_idx in self.class_to_idx.items():
            class_dir = class_dirs.get(class_key)
            if not class_dir:
                continue
            for file_name in os.listdir(class_dir):
                if file_name.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
                    self.samples.append((os.path.join(class_dir, file_name), class_idx))

        if len(self.samples) == 0:
            available_dirs = [d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))]
            raise ValueError(
                "No images found for class folders 'open' and 'closed'. "
                f"Root: {root_dir}. Available folders: {available_dirs}"
            )

        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ])

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        img = self.transform(img)
        label = torch.tensor(label, dtype=torch.long)
        return img, label

In [5]:
class Generator(nn.Module):
    def __init__(self, z_dim: int, num_classes: int, img_channels: int = 3, feature_g: int = 64):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, z_dim)

        self.net = nn.Sequential(
            nn.ConvTranspose2d(z_dim * 2, feature_g * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(feature_g * 8),
            nn.ReLU(True),

            nn.ConvTranspose2d(feature_g * 8, feature_g * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_g * 4),
            nn.ReLU(True),

            nn.ConvTranspose2d(feature_g * 4, feature_g * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_g * 2),
            nn.ReLU(True),

            nn.ConvTranspose2d(feature_g * 2, feature_g, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_g),
            nn.ReLU(True),

            nn.ConvTranspose2d(feature_g, img_channels, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, noise: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        label_vec = self.label_emb(labels)
        x = torch.cat([noise, label_vec], dim=1)
        x = x.unsqueeze(2).unsqueeze(3)
        return self.net(x)

In [6]:
class Discriminator(nn.Module):
    def __init__(self, num_classes: int, img_channels: int = 3, feature_d: int = 64):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, IMAGE_SIZE * IMAGE_SIZE)

        self.net = nn.Sequential(
            nn.Conv2d(img_channels + 1, feature_d, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, True),

            nn.Conv2d(feature_d, feature_d * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_d * 2),
            nn.LeakyReLU(0.2, True),

            nn.Conv2d(feature_d * 2, feature_d * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_d * 4),
            nn.LeakyReLU(0.2, True),

            nn.Conv2d(feature_d * 4, feature_d * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_d * 8),
            nn.LeakyReLU(0.2, True),

            nn.Conv2d(feature_d * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, img: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        label_map = self.label_emb(labels).view(-1, 1, IMAGE_SIZE, IMAGE_SIZE)
        x = torch.cat([img, label_map], dim=1)
        out = self.net(x)
        return out.view(-1)


In [7]:
def save_generated_grid(generator: nn.Module, epoch: int, fixed_noise: torch.Tensor, fixed_labels: torch.Tensor) -> None:
    generator.eval()
    with torch.no_grad():
        fake = generator(fixed_noise, fixed_labels).detach().cpu()
        fake = (fake + 1) / 2
        grid = utils.make_grid(fake, nrow=10)
        utils.save_image(grid, os.path.join(OUTPUT_DIR, f"epoch_{epoch:03d}.png"))
    generator.train()


def generate_images(generator: nn.Module, class_name: str, label_idx: int, num_images: int = 10) -> None:
    generator.eval()
    label_tensor = torch.full((num_images,), label_idx, dtype=torch.long, device=device)
    noise = torch.randn(num_images, Z_DIM, device=device)
    with torch.no_grad():
        fake = generator(noise, label_tensor).detach().cpu()
        fake = (fake + 1) / 2

    out_dir = os.path.join(OUTPUT_DIR, f"{class_name}_samples")
    os.makedirs(out_dir, exist_ok=True)
    for i in range(num_images):
        utils.save_image(fake[i], os.path.join(out_dir, f"{class_name}_{i+1}.png"))
    generator.train()

In [8]:
def train() -> None:
    dataset = BlinkDataset(DATA_DIR, IMAGE_SIZE)
    train_size = int(len(dataset) * TRAIN_SPLIT)
    val_size = len(dataset) - train_size
    train_set, _ = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(SEED))

    loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)

    generator = Generator(Z_DIM, NUM_CLASSES).to(device)
    discriminator = Discriminator(NUM_CLASSES).to(device)

    criterion = nn.BCELoss()
    opt_g = torch.optim.Adam(generator.parameters(), lr=LR, betas=BETAS)
    opt_d = torch.optim.Adam(discriminator.parameters(), lr=LR, betas=BETAS)

    fixed_noise = torch.randn(20, Z_DIM, device=device)
    fixed_labels = torch.tensor([0] * 10 + [1] * 10, dtype=torch.long, device=device)

    print(f"Training on {device} with {len(train_set)} images")

    for epoch in range(1, EPOCHS + 1):
        for real_imgs, labels in loader:
            real_imgs = real_imgs.to(device)
            labels = labels.to(device)
            batch_size = real_imgs.size(0)

            real_targets = torch.ones(batch_size, device=device)
            fake_targets = torch.zeros(batch_size, device=device)

            # Train Discriminator
            noise = torch.randn(batch_size, Z_DIM, device=device)
            fake_labels = torch.randint(0, NUM_CLASSES, (batch_size,), device=device)
            fake_imgs = generator(noise, fake_labels)

            d_real = discriminator(real_imgs, labels)
            d_fake = discriminator(fake_imgs.detach(), fake_labels)

            loss_d_real = criterion(d_real, real_targets)
            loss_d_fake = criterion(d_fake, fake_targets)
            loss_d = (loss_d_real + loss_d_fake) / 2

            opt_d.zero_grad()
            loss_d.backward()
            opt_d.step()

            # Train Generator
            output = discriminator(fake_imgs, fake_labels)
            loss_g = criterion(output, real_targets)

            opt_g.zero_grad()
            loss_g.backward()
            opt_g.step()

        print(f"Epoch [{epoch}/{EPOCHS}]  Loss D: {loss_d.item():.4f}  Loss G: {loss_g.item():.4f}")

        if epoch % SAMPLE_EVERY == 0 or epoch == 1:
            save_generated_grid(generator, epoch, fixed_noise, fixed_labels)

    torch.save(generator.state_dict(), os.path.join(OUTPUT_DIR, "generator.pth"))
    torch.save(discriminator.state_dict(), os.path.join(OUTPUT_DIR, "discriminator.pth"))

    # Generate at least 10 images for each class
    generate_images(generator, "Open", 0, 10)
    generate_images(generator, "Closed", 1, 10)

    end_dt = datetime.now()
    total_seconds = time.time() - start_ts
    print("Done. Generated images are saved in cgan_outputs/")

In [13]:
train()

Training started at: 2026-04-11 09:49:03
Training on cpu with 37484 images
Epoch [1/12]  Loss D: 0.0627  Loss G: 4.1795
Epoch [2/12]  Loss D: 0.0238  Loss G: 7.2151
Epoch [3/12]  Loss D: 0.5198  Loss G: 3.7779
Epoch [4/12]  Loss D: 0.7294  Loss G: 5.8810
Epoch [5/12]  Loss D: 0.3837  Loss G: 3.0014
Epoch [6/12]  Loss D: 0.3809  Loss G: 3.3121
Epoch [7/12]  Loss D: 0.3553  Loss G: 2.9506
Epoch [8/12]  Loss D: 0.2899  Loss G: 2.5108
Epoch [9/12]  Loss D: 0.1117  Loss G: 3.7833
Epoch [10/12]  Loss D: 0.1150  Loss G: 4.1219
Epoch [11/12]  Loss D: 0.1101  Loss G: 3.4482
Epoch [12/12]  Loss D: 0.1565  Loss G: 3.0735
Done. Generated images are saved in cgan_outputs/
